In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder("cats_and_dogs_filtered/train", transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

class CatDogCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*56*56,128),
            nn.ReLU(),
            nn.Linear(128,1)
        )

    def forward(self,x):
        x = self.conv(x)
        return self.fc(x)

def compute_weight_norm(model):
    total = 0
    for p in model.parameters():
        total += torch.sum(p**2)
    return torch.sqrt(total).item()


def count_zero_weights(model, threshold=1e-5):
    count, total = 0, 0
    for p in model.parameters():
        count += torch.sum(torch.abs(p) < threshold).item()
        total += p.numel()
    return count, total

def train_model(reg_type=None, method=None, lambda_reg=1e-4, epochs=5):
    model = CatDogCNN().to(device)
    criterion = nn.BCEWithLogitsLoss()
    if method == "weight_decay":
        optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=lambda_reg)
    else:
        optimizer = optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(epochs):
        running_loss = 0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            if method == "manual":
                reg = 0
                for param in model.parameters():
                    if reg_type == "L2":
                        reg += torch.sum(param**2)
                    elif reg_type == "L1":
                        reg += torch.sum(torch.abs(param))
                loss = loss + lambda_reg * reg
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}")

    norm = compute_weight_norm(model)
    zeros, total = count_zero_weights(model)

    return norm, zeros, total

l2_decay_norm, _, _ = train_model("L2","weight_decay")
l2_manual_norm, _, _ = train_model("L2","manual")

l1_decay_norm, l1_zeros, total = train_model("L1","weight_decay")
l1_manual_norm, l1_manual_zeros, _ = train_model("L1","manual")

Epoch 1, Loss: 53.7437
Epoch 2, Loss: 43.4951
Epoch 3, Loss: 43.2888
Epoch 4, Loss: 41.0779
Epoch 5, Loss: 37.2822
Epoch 1, Loss: 61.7149
Epoch 2, Loss: 45.1700
Epoch 3, Loss: 44.6015
Epoch 4, Loss: 43.8766
Epoch 5, Loss: 43.5850
Epoch 1, Loss: 48.2720
Epoch 2, Loss: 42.9939
Epoch 3, Loss: 41.3487
Epoch 4, Loss: 37.3559
Epoch 5, Loss: 32.3946
Epoch 1, Loss: 115.9436
Epoch 2, Loss: 64.7869
Epoch 3, Loss: 63.6656
Epoch 4, Loss: 63.5044
Epoch 5, Loss: 63.6849


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.ImageFolder("cats_and_dogs_filtered/train", transform=transform)
val_data = datasets.ImageFolder("cats_and_dogs_filtered/validation", transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)

class CustomDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p

    def forward(self, x):
        if self.training:
            mask = torch.bernoulli(torch.ones_like(x) * (1 - self.p))
            return (x * mask) / (1 - self.p)   # scale to keep expected value same
        else:
            return x

class CatDogCNN(nn.Module):
    def __init__(self, dropout_type=None):
        super().__init__()
        if dropout_type == "torch":
            dropout = nn.Dropout(0.5)
        elif dropout_type == "custom":
            dropout = CustomDropout(0.5)
        else:
            dropout = nn.Identity()

        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*32*32,128),
            nn.ReLU(),
            dropout,
            nn.Linear(128,2)
        )

    def forward(self,x):
        x = self.features(x)
        x = self.classifier(x)
        return x
    
def train_model(model, epochs=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        
        for x,y in train_loader:
            x,y = x.to(device), y.to(device)

            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out,y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        print(f"Epoch {epoch+1} Train Loss: {train_loss/len(train_loader):.4f}")

        evaluate(model)

def evaluate(model):
    device = next(model.parameters()).device
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for x,y in val_loader:
            x,y = x.to(device), y.to(device)
            out = model(x)
            preds = torch.argmax(out,1)

            correct += (preds==y).sum().item()
            total += y.size(0)

    acc = correct/total
    print("Validation Accuracy:", acc)

model_no_dropout = CatDogCNN(dropout_type=None)
train_model(model_no_dropout)

model_torch_dropout = CatDogCNN(dropout_type="torch")
train_model(model_torch_dropout)

model_custom_dropout = CatDogCNN(dropout_type="custom")
train_model(model_custom_dropout)

/home/mca/anaconda3/lib/python3.11/site-packages/torch/cuda/__init__.py:184: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Epoch 1 Train Loss: 0.7336
Validation Accuracy: 0.555
Epoch 2 Train Loss: 0.6395
Validation Accuracy: 0.673
Epoch 3 Train Loss: 0.5542
Validation Accuracy: 0.676
Epoch 4 Train Loss: 0.4669
Validation Accuracy: 0.689
Epoch 5 Train Loss: 0.3616
Validation Accuracy: 0.681
Epoch 1 Train Loss: 0.7613
Validation Accuracy: 0.54
Epoch 2 Train Loss: 0.6564
Validation Accuracy: 0.64
Epoch 3 Train Loss: 0.6003
Validation Accuracy: 0.655
Epoch 4 Train Loss: 0.5608
Validation Accuracy: 0.689
Epoch 5 Train Loss: 0.5116
Validation Accuracy: 0.669
Epoch 1 Train Loss: 0.7151
Validation Accuracy: 0.654
Epoch 2 Train Loss: 0.6243
Validation Accuracy: 0.682
Epoch 3 Train Loss: 0.5550
Validation Accuracy: 0.678
Epoch 4 Train Loss: 0.4966
Validation Accuracy: 0.706
Epoch 5 Train Loss: 0.4224
Validation Accuracy: 0.701


In [5]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder("cats_and_dogs_filtered/train", transform=transform)
val_dataset = datasets.ImageFolder("cats_and_dogs_filtered/validation", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

class CatDogCNN(nn.Module):
    def __init__(self, dropout_type=None):
        super().__init__()
        if dropout_type == "torch":
            dropout = nn.Dropout(0.5)
        elif dropout_type == "custom":
            # Assuming CustomDropout is defined elsewhere in your code
            dropout = CustomDropout(0.5)
        else:
            dropout = nn.Identity()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), # 224 -> 112

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)  # 112 -> 56
        )

        # 64 channels * 56 height * 56 width = 200,704
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 56 * 56, 128), 
            nn.ReLU(),
            dropout,
            nn.Linear(128, 1) # Changed to 1 for BCEWithLogitsLoss
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def validate(model, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

    return total_loss / len(val_loader)

device = 'cpu'

model_no_es = CatDogCNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_no_es.parameters(), lr=0.001)

num_epochs = 10

for epoch in range(num_epochs):
    model_no_es.train()
    train_loss = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        outputs = model_no_es(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    val_loss = validate(model_no_es, criterion)
    print(f"Epoch {epoch+1} Train Loss:{train_loss:.4f}  Val Loss:{val_loss:.4f}")

model_es = CatDogCNN().to(device)
optimizer = optim.Adam(model_es.parameters(), lr=0.001)

best_val_loss = float("inf")
patience = 2
counter = 0

for epoch in range(20):
    model_es.train()
    train_loss = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        outputs = model_es(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    val_loss = validate(model_es, criterion)
    print(f"Epoch {epoch+1} Train:{train_loss:.4f}  Val:{val_loss:.4f}")
    # Early stopping condition
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
    else:
        counter += 1
    if counter >= patience:
        print("Early stopping triggered")
        break

Epoch 1 Train Loss:61.4791  Val Loss:0.6927
Epoch 2 Train Loss:43.6446  Val Loss:0.6924
Epoch 3 Train Loss:43.5771  Val Loss:0.6920
Epoch 4 Train Loss:43.3870  Val Loss:0.6908
Epoch 5 Train Loss:42.6135  Val Loss:0.6938
Epoch 6 Train Loss:41.4404  Val Loss:0.6795
Epoch 7 Train Loss:36.9485  Val Loss:0.6836
Epoch 8 Train Loss:28.4008  Val Loss:0.7143
Epoch 9 Train Loss:19.4254  Val Loss:0.8919
Epoch 10 Train Loss:8.8655  Val Loss:1.0346
Epoch 1 Train:54.1215  Val:0.6934
Epoch 2 Train:43.4117  Val:0.6863
Epoch 3 Train:40.9210  Val:0.6663
Epoch 4 Train:31.9332  Val:0.6329
Epoch 5 Train:19.1306  Val:0.7882
Epoch 6 Train:8.5448  Val:1.1092
Early stopping triggered
